# W3-D: Overlap-Controlled Class-Level Cross-Evaluation

This notebook documents the workflow for evaluating the four final checkpoints on a clean common test subset.

Goal: compare `rgb_standard`, `rgb_spatial`, `ms_standard`, and `ms_spatial` checkpoints at class level and Macro-F1 after controlling for cross-protocol split overlap.


## Experiment Matrix

This workflow uses four final checkpoints:

| Model | Train Split | Config |
|---|---|---|
| RGB ResNet-50 | standard | `configs/rgb_standard.yaml` |
| RGB ResNet-50 | spatial | `configs/rgb_spatial.yaml` |
| MS ResNet-50 | standard | `configs/ms_standard.yaml` |
| MS ResNet-50 | spatial | `configs/ms_spatial.yaml` |

Each checkpoint is evaluated on the overlap-controlled clean common test subset. The final output compares the four checkpoints by class-level F1 and Macro-F1.

Hyperparameters are reused from the prior spatial grid search:

- RGB: `learning_rate=1e-4`, `weight_decay=1e-5`
- MS: `learning_rate=1e-4`, `weight_decay=1e-3`
- All models: `pretrained=true`, `epochs=10`, `batch_size=32`, `seed=42`
- MS models use `normalization=zscore`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(PROJECT_ROOT)
assert (PROJECT_ROOT / "pyproject.toml").exists(), "Run this notebook from the repo root or notebooks/ directory."

## 1. Write The Four Training Configs

Run this cell to create or refresh the four config files used for this experiment.

In [ ]:
CONFIGS = {
    "configs/rgb_standard.yaml": """modality: rgb
split_type: standard
model_input_mode: direct
pretrained: true
epochs: 10
batch_size: 32
num_workers: 4
learning_rate: 0.0001
weight_decay: 0.00001
seed: 42
device: auto
pin_memory: true
""",
    "configs/rgb_spatial.yaml": """modality: rgb
split_type: spatial
model_input_mode: direct
pretrained: true
epochs: 10
batch_size: 32
num_workers: 4
learning_rate: 0.0001
weight_decay: 0.00001
seed: 42
device: auto
pin_memory: true
""",
    "configs/ms_standard.yaml": """modality: ms
split_type: standard
model_input_mode: direct
pretrained: true
normalization: zscore
epochs: 10
batch_size: 32
num_workers: 4
learning_rate: 0.0001
weight_decay: 0.001
seed: 42
device: auto
pin_memory: true
""",
    "configs/ms_spatial.yaml": """modality: ms
split_type: spatial
model_input_mode: direct
pretrained: true
normalization: zscore
epochs: 10
batch_size: 32
num_workers: 4
learning_rate: 0.0001
weight_decay: 0.001
seed: 42
device: auto
pin_memory: true
""",
}

for rel_path, text in CONFIGS.items():
    path = PROJECT_ROOT / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    print(f"wrote {rel_path}")

## 2. Verify Configs And Expected Run Names

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from satx.engine import load_training_config

config_paths = [
    "configs/rgb_standard.yaml",
    "configs/rgb_spatial.yaml",
    "configs/ms_standard.yaml",
    "configs/ms_spatial.yaml",
]

for rel_path in config_paths:
    cfg = load_training_config(PROJECT_ROOT / rel_path)
    print(f"{rel_path}: {cfg.run_name}")

## 3. Train The Four Models

Set `RUN_TRAINING = True` only when you are ready to train. Training can take a while. Each run writes `best.pt`, `last.pt`, and `metrics_history.json` under `outputs/runs/<run_name>/`.

The training script selects `best.pt` by validation macro-F1.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    logs_dir = PROJECT_ROOT / "outputs" / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    for rel_path in config_paths:
        name = Path(rel_path).stem
        log_path = logs_dir / f"{name}.log"
        print(f"Running {rel_path}")
        with log_path.open("w", encoding="utf-8") as log_file:
            result = subprocess.run(
                [sys.executable, "scripts/train.py", rel_path],
                cwd=PROJECT_ROOT,
                stdout=log_file,
                stderr=subprocess.STDOUT,
                text=True,
            )
        if result.returncode != 0:
            raise RuntimeError(f"Training failed for {rel_path}; see {log_path}")
        print(f"Finished {rel_path}; log: {log_path}")
else:
    print("Training is disabled. Set RUN_TRAINING = True to train from this notebook.")

Terminal alternative:

```bash
mkdir -p outputs/logs
set -e
for cfg in configs/rgb_standard.yaml configs/rgb_spatial.yaml configs/ms_standard.yaml configs/ms_spatial.yaml; do
  name=$(basename "$cfg" .yaml)
  echo "Running $cfg"
  python scripts/train.py "$cfg" 2>&1 | tee "outputs/logs/${name}.log"
done
```

## 4. Inspect Training Outputs

In [ ]:
run_dirs = []
for rel_path in config_paths:
    cfg = load_training_config(PROJECT_ROOT / rel_path)
    run_dir = cfg.run_dir()
    run_dirs.append(run_dir)
    print("\n", run_dir.name)
    print("best.pt exists:", (run_dir / "best.pt").exists())
    history_path = run_dir / "metrics_history.json"
    print("metrics_history.json exists:", history_path.exists())
    if history_path.exists():
        history = json.loads(history_path.read_text(encoding="utf-8"))
        print("best_epoch:", history.get("best_epoch"))
        print("best_val_macro_f1:", history.get("best_val_macro_f1"))
        print("best_val_accuracy:", history.get("best_val_accuracy"))

## 5. Create The Cross-Evaluation Manifest

The manifest tells `scripts/overlap_controlled_class_level_cross_eval.py` which checkpoints to evaluate.

In [ ]:
manifest_path = PROJECT_ROOT / "outputs" / "summary" / "runs_manifest.csv"
manifest_path.parent.mkdir(parents=True, exist_ok=True)

manifest_text = """run_name,modality,train_split,input_mode,pretrained,seed,checkpoint_path
rgb_standard,rgb,standard,direct,true,42,outputs/runs/resnet50_rgb_standard_direct_pretrained_seed42/best.pt
rgb_spatial,rgb,spatial,direct,true,42,outputs/runs/resnet50_rgb_spatial_direct_pretrained_seed42/best.pt
ms_standard,ms,standard,direct,true,42,outputs/runs/resnet50_ms_standard_direct_pretrained_zscore_seed42/best.pt
ms_spatial,ms,spatial,direct,true,42,outputs/runs/resnet50_ms_spatial_direct_pretrained_zscore_seed42/best.pt
"""
manifest_path.write_text(manifest_text, encoding="utf-8")
print(manifest_path)
print(manifest_path.read_text(encoding="utf-8"))

## 6. Run Overlap-Controlled Class-Level Cross-Evaluation

This evaluates each checkpoint on the clean common test subset obtained by removing cross-protocol train/validation overlap from the requested test splits.

Set `RUN_CROSS_EVAL = True` after the four `best.pt` files exist.


In [ ]:
RUN_CROSS_EVAL = True

if RUN_CROSS_EVAL:
    result = subprocess.run(
        [
            sys.executable,
            "scripts/overlap_controlled_class_level_cross_eval.py",
            "--manifest",
            "outputs/summary/runs_manifest.csv",
            "--test-splits",
            "standard",
            "spatial",
        ],
        cwd=PROJECT_ROOT,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError("Overlap-controlled class-level cross-evaluation failed.")
else:
    print("Evaluation is disabled. Set RUN_CROSS_EVAL = True after training finishes.")


Terminal alternative:

```bash
python scripts/overlap_controlled_class_level_cross_eval.py \
  --manifest outputs/summary/runs_manifest.csv \
  --test-splits standard spatial
```


## 7. Load The Main Results

In [ ]:
import csv

results_path = PROJECT_ROOT / "results" / "overlap_controlled_class_level_cross_eval" / "class_level_f1_overlap_controlled_test_set.csv"
if results_path.exists():
    with results_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    for row in rows"],
            "rgb_spatial_f1=", row["rgb_spatial_f1"],
            "ms_standard_f1=", row["ms_standard_f1"],
            "ms_spatial_f1=", row["ms_spatial_f:
        print(
            row["class_name"],
            "rgb_standard_f1=", row["rgb_standard_f11"],
        )
else:
    print(f"No results yet: {results_path}")


## Deliverables

This experiment writes one overlap-controlled class-level F1 table and one plot:

- `results/overlap_controlled_class_level_cross_eval/class_level_f1_overlap_controlled_test_set.csv`
- `results/overlap_controlled_class_level_cross_eval/class_level_f1_overlap_controlled_test_set.png`

The table compares `rgb_standard`, `rgb_spatial`, `ms_standard`, and `ms_spatial` on the clean common test subset. The final `Macro-F1` row averages F1 across classes.
